Position interpolation from Riot's 60-second timeline snapshots.

In [1]:
import numpy as np

known_positions = [
    {"t": 90_000,  "x": 3850, "y": 8000},   # Blue buff (1:30)
    {"t": 150_000, "x": 2100, "y": 8400},   # Gromp    (2:30)
    {"t": 210_000, "x": 3750, "y": 6400},   # Wolves   (3:30)
]

def linear_interpolate(p1, p2, target_t):
    t = (target_t - p1["t"]) / (p2["t"] - p1["t"])
    return {
        "t": target_t,
        "x": p1["x"] + t * (p2["x"] - p1["x"]),
        "y": p1["y"] + t * (p2["y"] - p1["y"]),
    }

interpolated = []
for i in range(len(known_positions) - 1):
    p1, p2 = known_positions[i], known_positions[i + 1]
    for t in range(p1["t"], p2["t"], 10_000):
        interpolated.append(linear_interpolate(p1, p2, t))
interpolated.append(known_positions[-1])

print(f"{len(known_positions)} known positions -> {len(interpolated)} interpolated points")
for p in interpolated:
    mins, secs = divmod(p['t'] // 1000, 60)
    print(f"  {mins}:{secs:02d}  ({p['x']:.0f}, {p['y']:.0f})")

3 known positions -> 13 interpolated points
  1:30  (3850, 8000)
  1:40  (3558, 8067)
  1:50  (3267, 8133)
  2:00  (2975, 8200)
  2:10  (2683, 8267)
  2:20  (2392, 8333)
  2:30  (2100, 8400)
  2:40  (2375, 8067)
  2:50  (2650, 7733)
  3:00  (2925, 7400)
  3:10  (3200, 7067)
  3:20  (3475, 6733)
  3:30  (3750, 6400)


The path from Gromp to Wolves cuts straight through the jungle wall between them. A* pathfinding on the navgrid routes around it.

In [2]:
from analytics.pathfinding import NavGrid, AStarPathfinder

navgrid = NavGrid()
pathfinder = AStarPathfinder(navgrid)

start = navgrid.game_to_grid(2100, 8400)  # Gromp
end   = navgrid.game_to_grid(3750, 6400)  # Wolves

path = pathfinder.find_path(start, end)
linear_dist = np.sqrt((3750 - 2100)**2 + (6400 - 8400)**2)
path_dist = sum(
    np.sqrt((path[i+1].x - path[i].x)**2 + (path[i+1].y - path[i].y)**2)
    for i in range(len(path) - 1)
) * 50  # grid cells -> game units

print(f"Linear distance:     {linear_dist:.0f} game units")
print(f"A* path distance:    {path_dist:.0f} game units ({len(path)} waypoints)")
print(f"Detour factor:       {path_dist / linear_dist:.2f}x")

Linear distance:     2561 game units
A* path distance:    3074 game units (52 waypoints)
Detour factor:       1.20x


Using the interpolated positions and clear event timestamps, we can compare actual clear speed to par time.

In [3]:
import json

with open("data/par_times.json") as f:
    par_times = {k: v for k, v in json.load(f).items()
                 if isinstance(v, (int, float))}

clear_path = "Blue-Gromp-Wolves-Raptors-Red-Krugs"
actual_time = 272  # seconds, from a real Viego game
par = par_times[clear_path]

print(f"Path:        {clear_path}")
print(f"Par time:    {par // 60}:{par % 60:02d} ({par}s)")
print(f"Actual time: {actual_time // 60}:{actual_time % 60:02d} ({actual_time}s)")
print(f"Tempo loss:  +{actual_time - par}s")

Path:        Blue-Gromp-Wolves-Raptors-Red-Krugs
Par time:    4:20 (260s)
Actual time: 4:32 (272s)
Tempo loss:  +12s
